# One-click Epicollect5 daily update

Run **Kernel > Restart & Run All** once each day. This notebook performs the complete workflow in order:

1. Preflight paths, Git state, configuration, and the GeoPackage lock.
2. Fetch new/edited entries and atomically write CSV, Excel, and the GeoPackage.
3. ID-join detections, rebuild detection-box-centroid crosses, and publish HTML.
4. Refresh category, larval, join/confusion-matrix, and dry-to-water analyses.
5. Resumably download originals, blur small faces with the tiled area filter, and push only generated JPGs.
6. Rebuild and publish HTML again so new photos replace placeholders.
7. Reconcile local files, analysis outputs, Git, and the deployed page.

> Before running: close `epicollect5.gpkg` in ArcGIS Pro and QGIS. The preflight stops safely if it is still locked. ArcGIS Online remains unchanged while `AGOL_SYNC = False`.


In [ ]:
from pathlib import Path
from datetime import datetime
from collections import deque
import ctypes
import hashlib
import json
import os
import re
import subprocess
import sys
import time

import geopandas as gpd
import pandas as pd
import requests
from IPython.display import display

EXPECTED_REPO = Path(
    r"D:\OneDrive_Emory\OneDrive - Emory\Research_doc\larvae\code"
    r"\download_Epicollect5\larvae-photos"
)
candidates = [Path.cwd(), Path.cwd() / "larvae-photos", EXPECTED_REPO]
REPO = next((path.resolve() for path in candidates
             if (path / "daily_update.py").is_file()), None)
if REPO is None:
    raise FileNotFoundError("Could not locate larvae-photos/daily_update.py")
PROJECT = REPO.parent
OUTPUT = PROJECT / "output"
SOURCE_NOTEBOOK = PROJECT / "download_epicollect5.ipynb"
ANALYSIS_NOTEBOOK = PROJECT / "survey_distribution_summary.ipynb"
CSV_PATH = OUTPUT / "tables" / "gps_and_pictures.csv"
EXCEL_PATH = OUTPUT / "epicollect5_tables.xlsx"
GPKG_PATH = OUTPUT / "epicollect5.gpkg"
ORIGINAL_DIR = OUTPUT / "photos"
BLURRED_DIR = REPO / "photos"
HTML_PATH = REPO / "docs" / "index.html"
SITE_URL = "https://gladcolor.github.io/larvae-photos/"
EXPECTED_BRANCH = "main"
VERIFY_DEPLOYMENT = True
DEPLOY_TIMEOUT_SECONDS = 300

print(f"repository : {REPO}")
print(f"python     : {sys.executable}")
print(f"started    : {datetime.now():%Y-%m-%d %H:%M:%S}")


In [ ]:
def run(command, cwd=REPO, check=True, capture=True):
    command = [str(value) for value in command]
    shown = subprocess.list2cmdline(command)
    print(f"$ {shown}")
    done = subprocess.run(
        command, cwd=str(cwd), text=True,
        capture_output=capture,
    )
    if capture:
        if done.stdout.strip():
            print(done.stdout.rstrip())
        if done.stderr.strip():
            print(done.stderr.rstrip(), file=sys.stderr)
    if check and done.returncode != 0:
        raise RuntimeError(f"command failed with exit code {done.returncode}: {shown}")
    return done


def git(*args, check=True):
    return run(["git", *args], check=check, capture=True).stdout.strip()


def normalize_site_id(value):
    if value is None or pd.isna(value):
        return ""
    text = str(value).strip().upper()
    return re.sub(r"^([+-]?\d+)\.0+$", r"\1", text)


def find_field(columns, suffix):
    suffix = suffix.lower()
    return next((name for name in columns
                 if name.lower() == suffix or name.lower().endswith("_" + suffix)), None)


def require_exclusive_windows_access(path):
    if not path.exists() or os.name != "nt":
        return
    from ctypes import wintypes
    create_file = ctypes.WinDLL("kernel32", use_last_error=True).CreateFileW
    create_file.argtypes = [wintypes.LPCWSTR, wintypes.DWORD, wintypes.DWORD,
                            wintypes.LPVOID, wintypes.DWORD, wintypes.DWORD,
                            wintypes.HANDLE]
    create_file.restype = wintypes.HANDLE
    handle = create_file(str(path), 0x80000000 | 0x40000000, 0, None, 3, 0x80, None)
    invalid = ctypes.c_void_p(-1).value
    if handle == invalid:
        error = ctypes.get_last_error()
        raise RuntimeError(
            f"{path.name} is locked (Windows error {error}). "
            "Close it in ArcGIS Pro/QGIS, then restart this notebook."
        )
    ctypes.WinDLL("kernel32", use_last_error=True).CloseHandle(handle)


def html_rows(text):
    return re.findall(r'<tr data-id="([^"]*)"[^>]*>(.*?)</tr>', text, re.S)


## 1. Preflight

This cell makes no data changes. It refuses to start if a concurrent run, GIS lock, staged Git work, generated-output edits, wrong branch, stale remote, or unsafe notebook setting is detected.


In [ ]:
required = [SOURCE_NOTEBOOK, ANALYSIS_NOTEBOOK, REPO / "daily_update.py",
            REPO / "face_blur.py", REPO / "make_contact_sheet.py"]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

require_exclusive_windows_access(GPKG_PATH)

branch = git("rev-parse", "--abbrev-ref", "HEAD")
if branch != EXPECTED_BRANCH:
    raise RuntimeError(f"Expected branch {EXPECTED_BRANCH!r}, found {branch!r}")
origin = git("remote", "get-url", "origin")
if "gladcolor/larvae-photos" not in origin.lower().replace(".git", ""):
    raise RuntimeError(f"Unexpected Git remote: {origin}")

staged = git("diff", "--cached", "--name-only")
if staged:
    raise RuntimeError("Unstage these files before the daily run:\n" + staged)
docs_dirty = git("status", "--porcelain", "--untracked-files=all",
                 "--", "docs")
if docs_dirty:
    raise RuntimeError("Generated docs already have changes:\n" + docs_dirty)
tracked_photo_dirty = git("diff", "--name-only", "--",
                          ":(glob)photos/*.jpg")
if tracked_photo_dirty:
    raise RuntimeError(
        "Tracked public photos were modified; reconcile them first:\n"
        + tracked_photo_dirty
    )
resumable = git("ls-files", "--others", "--exclude-standard",
                "--", ":(glob)photos/*.jpg").splitlines()
unexpected_resumable = [path for path in resumable
                        if not (ORIGINAL_DIR / Path(path).name).is_file()]
if unexpected_resumable:
    raise RuntimeError(
        "Untracked public JPGs have no matching private original:\n"
        + "\n".join(unexpected_resumable[:20])
    )
if resumable:
    print(f"resuming {len(resumable)} previously blurred, uncommitted JPG(s)")

git("fetch", "origin", EXPECTED_BRANCH)
local_head = git("rev-parse", "HEAD")
remote_head = git("rev-parse", f"origin/{EXPECTED_BRANCH}")
if local_head != remote_head:
    remote_is_ancestor = run(
        ["git", "merge-base", "--is-ancestor",
         f"origin/{EXPECTED_BRANCH}", "HEAD"],
        check=False, capture=True,
    ).returncode == 0
    ahead_paths = (git("diff", "--name-only",
                       f"origin/{EXPECTED_BRANCH}..HEAD").splitlines()
                   if remote_is_ancestor else [])
    safe_ahead = bool(ahead_paths) and all(
        path.replace("\\", "/").startswith("docs/")
        or re.fullmatch(r"photos/[^/]+[.]jpg", path.replace("\\", "/"), re.I)
        for path in ahead_paths
    )
    if not safe_ahead:
        raise RuntimeError(
            f"Local HEAD {local_head[:8]} differs from origin/{EXPECTED_BRANCH} "
            f"{remote_head[:8]}. Reconcile Git before running."
        )
    print(f"recovering {len(ahead_paths)} generated path(s) from a prior push failure")
    pushed = False
    for attempt in range(1, 4):
        pushed = run(
            ["git", "push", "origin", f"HEAD:{EXPECTED_BRANCH}"],
            check=False, capture=True,
        ).returncode == 0
        if pushed:
            break
        time.sleep(5 * attempt)
    if not pushed:
        raise RuntimeError("Could not recover the prior generated-output push")
    git("fetch", "origin", EXPECTED_BRANCH)
    remote_head = git("rev-parse", f"origin/{EXPECTED_BRANCH}")
    if local_head != remote_head:
        raise RuntimeError("Recovered push did not update the expected remote head")

source = json.loads(SOURCE_NOTEBOOK.read_text(encoding="utf-8"))
settings = "".join(source["cells"][2]["source"])
for required_setting in ["DOWNLOAD_PHOTOS = True", "MAX_PHOTOS = None",
                         "BLUR_FACES = True", "GIT_PUSH = True",
                         'PHOTO_URL_MODE = "all"']:
    if required_setting not in settings:
        raise RuntimeError(f"Required safe setting not found: {required_setting}")

unrelated = git("status", "--short")
if unrelated:
    print("Unrelated worktree changes will be preserved:")
    print(unrelated)
print("PRE-FLIGHT PASSED")


## 2. Run the complete pipeline

The runner has its own cross-process lock. It publishes the refreshed data/site before the rate-limited media queue, then selectively publishes generated JPGs and performs a final site refresh. Photo downloads are resumable, so restarting after a network interruption is safe.


In [ ]:
RUN_STARTED = datetime.now()
START_HEAD = git("rev-parse", "HEAD")
log_path = OUTPUT / "daily_update.log"
log_offset = log_path.stat().st_size if log_path.exists() else 0
command = [sys.executable, "-u", str(REPO / "daily_update.py")]
print("$ " + subprocess.list2cmdline(command), flush=True)
tail = deque(maxlen=30)
child_env = os.environ.copy()
child_env["PYTHONIOENCODING"] = "utf-8"
child_env["PYTHONUNBUFFERED"] = "1"
process = subprocess.Popen(
    command, cwd=str(REPO), stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, text=True, encoding="utf-8",
    errors="replace", bufsize=1, env=child_env,
)
try:
    for line in process.stdout:
        print(line, end="", flush=True)
        clean_line = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", line.rstrip())
        tail.append(clean_line)
    returncode = process.wait()
except BaseException:
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
    raise
RUN_FINISHED = datetime.now()
elapsed = RUN_FINISHED - RUN_STARTED
if returncode != 0:
    appended_log = ""
    if log_path.exists():
        with log_path.open("rb") as handle:
            current_size = log_path.stat().st_size
            handle.seek(log_offset if current_size >= log_offset else 0)
            appended_log = handle.read().decode("utf-8", errors="replace")
            appended_log = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", appended_log)
    diagnostic_lines = list(tail) or appended_log.splitlines()[-30:]
    failed_stage = next(
        (line for line in reversed(diagnostic_lines) if "FAILED" in line),
        f"exit code {returncode}",
    )
    details = "\n".join(diagnostic_lines[-30:])
    raise RuntimeError(
        f"Daily update failed after {elapsed}: {failed_stage}\n\n"
        f"Recent pipeline output:\n{details}\n\nFull log: {log_path}"
    )
print(f"pipeline completed in {elapsed}")


## 3. Reconcile every output

Success requires table/geometry parity, exact original/blurred filename parity, no partial files, the expected detection fields, a clean generated-output scope, matching local/remote Git heads, and the deployed page matching the local HTML byte-for-byte.


In [ ]:
survey = pd.read_csv(CSV_PATH)
layer_name = gpd.list_layers(GPKG_PATH).iloc[0]["name"]
geo = gpd.read_file(GPKG_PATH, layer=layer_name)
excel_rows = len(pd.read_excel(EXCEL_PATH, sheet_name=0))
originals = {path.name for path in ORIGINAL_DIR.glob("*.jpg")}
blurred = {path.name for path in BLURRED_DIR.glob("*.jpg")}
patches = {path.name for path in (REPO / "docs" / "patches").glob("*.jpg")}
parts = list(OUTPUT.rglob("*.part")) + list(BLURRED_DIR.rglob("*.part"))
missing_report = OUTPUT / "photos_not_synced.txt"
not_synced = ({line.strip() for line in missing_report.read_text(encoding="utf-8").splitlines()
               if line.strip()} if missing_report.exists() else set())
photo_file_columns = [name for name in survey.columns if name.endswith("_file")]
referenced = {
    str(value).strip()
    for name in photo_file_columns
    for value in survey[name].dropna()
    if str(value).strip()
}
unaccounted_references = referenced - originals - not_synced
downloaded_but_reported_missing = originals & not_synced
unreferenced_missing = not_synced - referenced
remote_listing = subprocess.run(
    ["git", "ls-tree", "-r", "--name-only",
     f"origin/{EXPECTED_BRANCH}", "--", "photos"],
    cwd=str(REPO), capture_output=True, text=True, check=True,
).stdout
remote_photos = {Path(name).name for name in remote_listing.splitlines()
                 if Path(name).suffix.casefold() == ".jpg"}
analysis = json.loads(ANALYSIS_NOTEBOOK.read_text(encoding="utf-8"))
analysis_code = [cell for cell in analysis["cells"]
                 if cell.get("cell_type") == "code"]
analysis_errors = [output for cell in analysis_code
                   for output in cell.get("outputs", [])
                   if output.get("output_type") == "error"]

id_field = find_field(survey.columns, "habitat_id")
if not id_field:
    raise RuntimeError("Could not find the habitat ID field")
ids = survey[id_field].map(normalize_site_id)
xy = ids.str.startswith(("X", "Y"), na=False)
lat_field = find_field(survey.columns, "latitude")
lon_field = find_field(survey.columns, "longitude")
if not lat_field or not lon_field:
    raise RuntimeError("Could not find survey latitude/longitude fields")
coordinates = survey[[lon_field, lat_field]].apply(pd.to_numeric, errors="coerce")
mappable = coordinates.notna().all(axis=1)
mappable_rows = int(mappable.sum())
unmappable_rows = int((~mappable).sum())
mappable_xy = xy & mappable
unique_coordinate_pairs = len(coordinates.dropna().drop_duplicates())
survey_uuids = set(survey["ec5_uuid"].astype(str))
mappable_uuids = set(survey.loc[mappable, "ec5_uuid"].astype(str))
geo_uuids = set(geo["ec5_uuid"].astype(str))

html_bytes = HTML_PATH.read_bytes()
html_text = html_bytes.decode("utf-8")
rows = html_rows(html_text)
xy_bodies = [body for site_id, body in rows
             if site_id.upper().startswith(("X", "Y"))]

checks = {
    "CSV rows": len(survey),
    "Excel rows": excel_rows,
    "GeoPackage features": len(geo),
    "valid geometries": int((geo.geometry.notna() & ~geo.geometry.is_empty).sum()),
    "mappable CSV rows": mappable_rows,
    "unmappable CSV rows": unmappable_rows,
    "unique lon/lat pairs": unique_coordinate_pairs,
    "CSV unique UUIDs": len(survey_uuids),
    "GeoPackage unique UUIDs": len(geo_uuids),
    "referenced photos": len(referenced),
    "original JPGs": len(originals),
    "blurred JPGs": len(blurred),
    "remote JPGs": len(remote_photos),
    "original-only JPGs": len(originals - blurred),
    "blurred-only JPGs": len(blurred - originals),
    "unaccounted photo references": len(unaccounted_references),
    "downloaded but reported missing": len(downloaded_but_reported_missing),
    "unreferenced missing-photo lines": len(unreferenced_missing),
    "partial files": len(parts),
    "not-synced references": len(not_synced),
    "HTML rows": len(rows),
    "satellite patches": len(patches),
    "X/Y rows": int(xy.sum()),
    "mappable X/Y rows": int(mappable_xy.sum()),
    "X/Y imagery labels": sum("Imagery-detected habitat:" in body
                                    for body in xy_bodies),
    "field-observation labels": html_text.count("Field-observed habitat:"),
    "imagery-detection labels": html_text.count("Imagery-detected habitat:"),
    "comparison labels": html_text.count("Comparison:"),
    "off-view centroid notes": html_text.count("Detection centroid: Outside"),
    "analysis code cells": len(analysis_code),
    "analysis error outputs": len(analysis_errors),
}

assert checks["CSV rows"] == checks["Excel rows"]
assert checks["CSV rows"] == checks["CSV unique UUIDs"]
assert checks["CSV rows"] == checks["mappable CSV rows"] + checks["unmappable CSV rows"]
assert checks["mappable CSV rows"] == checks["GeoPackage features"] == checks["valid geometries"]
assert checks["mappable CSV rows"] == checks["GeoPackage unique UUIDs"]
assert geo_uuids == mappable_uuids
assert 0 < checks["unique lon/lat pairs"] <= checks["CSV rows"]
assert geo.crs is not None and geo.crs.to_epsg() == 4326
assert checks["mappable CSV rows"] == checks["HTML rows"] == checks["satellite patches"]
assert checks["HTML rows"] == checks["field-observation labels"]
assert checks["original JPGs"] == checks["blurred JPGs"]
assert originals == blurred == remote_photos
assert checks["original-only JPGs"] == checks["blurred-only JPGs"] == 0
assert checks["unaccounted photo references"] == 0
assert checks["downloaded but reported missing"] == 0
assert checks["unreferenced missing-photo lines"] == 0
assert checks["partial files"] == 0
assert checks["mappable X/Y rows"] == len(xy_bodies)
assert checks["X/Y imagery labels"] == 0
assert checks["comparison labels"] == checks["HTML rows"] - checks["mappable X/Y rows"]
assert analysis_code and all(cell.get("execution_count") is not None
                             for cell in analysis_code)
assert checks["analysis error outputs"] == 0
assert ANALYSIS_NOTEBOOK.stat().st_mtime >= RUN_STARTED.timestamp()

generated_dirty = git("status", "--porcelain", "--untracked-files=all",
                      "--", "docs", ":(glob)photos/*.jpg")
if generated_dirty:
    raise RuntimeError("Generated outputs remain uncommitted:\n" + generated_dirty)
if git("diff", "--cached", "--name-only"):
    raise RuntimeError("The Git index is not clean after publication")

FINAL_HEAD = git("rev-parse", "HEAD")
REMOTE_HEAD = git("ls-remote", "origin", f"refs/heads/{EXPECTED_BRANCH}").split()[0]
assert FINAL_HEAD == REMOTE_HEAD
committed_html_bytes = subprocess.run(
    ["git", "show", f"{FINAL_HEAD}:docs/index.html"],
    cwd=str(REPO), capture_output=True, check=True,
).stdout

deployment_verified = not VERIFY_DEPLOYMENT
if VERIFY_DEPLOYMENT:
    local_hash = hashlib.sha256(committed_html_bytes).hexdigest()
    deadline = time.time() + DEPLOY_TIMEOUT_SECONDS
    while time.time() < deadline:
        response = requests.get(
            SITE_URL, params={"verify": time.time_ns()}, timeout=30,
            headers={"Cache-Control": "no-cache"},
        )
        served_hash = hashlib.sha256(response.content).hexdigest()
        if response.status_code == 200 and served_hash == local_hash:
            deployment_verified = True
            break
        print("GitHub Pages has not refreshed yet; retrying in 10 seconds...")
        time.sleep(10)
    if not deployment_verified:
        raise RuntimeError(
            f"Git push succeeded, but {SITE_URL} did not match the local HTML "
            f"within {DEPLOY_TIMEOUT_SECONDS} seconds."
        )

summary = pd.DataFrame({"Result": checks}).rename_axis("Check")
display(summary)
print(f"local/remote commit : {FINAL_HEAD}")
print(f"deployed HTML       : {'verified' if deployment_verified else 'not requested'}")


In [ ]:
source_after = json.loads(SOURCE_NOTEBOOK.read_text(encoding="utf-8"))
settings_after = "".join(source_after["cells"][2]["source"])
agol_enabled = bool(re.search(r"^AGOL_SYNC\s*=\s*True",
                              settings_after, flags=re.M))
print("\nDAILY UPDATE COMPLETE")
print(f"finished : {datetime.now():%Y-%m-%d %H:%M:%S}")
print(f"site     : {SITE_URL}")
if not agol_enabled:
    print("note     : local GeoPackage updated; ArcGIS Online sync is disabled")
